# Notebook 5 — Final Comparison: Kids vs Adults Cross-Age Face Verification
### Loads results from Notebook 4 and compares the two life-stage categories directly

**Inputs:** `dataset_v2/kids/results/` and `dataset_v2/adults/results/`
(`genuine_scores.csv`, `impostor_scores.csv`, `eer_summary.csv` for each).

**What this notebook produces:**
1. Side-by-side genuine-score distributions (Kids vs Adults) per image condition
2. Combined ROC curves (Kids vs Adults) per condition
3. EER comparison bar chart
4. **Score-vs-age-gap comparison** (Kids vs Adults, overlaid) — the key question:
   *does verification degrade faster with age gap for children than for adults?*
5. A simple statistical significance check (Mann-Whitney U test) on genuine scores
6. A written conclusions template to fill in for your report


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

sns.set_style("whitegrid")

DATASET_ROOT = "./dataset_v2"
CATEGORIES = ["kids", "adults"]

genuine = {}
impostor = {}
eer = {}

for category in CATEGORIES:
    results_dir = os.path.join(DATASET_ROOT, category, "results")
    genuine[category] = pd.read_csv(os.path.join(results_dir, "genuine_scores.csv"))
    impostor[category] = pd.read_csv(os.path.join(results_dir, "impostor_scores.csv"))
    eer[category] = pd.read_csv(os.path.join(results_dir, "eer_summary.csv"))

VERSIONS = ["original", "gaussian_noise", "salt_pepper_noise", "synthetic_resize"]
COLORS = {"kids": "#8172B2", "adults": "#55A868"}

for c in CATEGORIES:
    print(c, "genuine:", len(genuine[c]), " impostor:", len(impostor[c]))


## 1. Genuine-score distributions: Kids vs Adults, per condition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, version_name in zip(axes.ravel(), VERSIONS):
    for category in CATEGORIES:
        scores = genuine[category][genuine[category]["version"] == version_name]["score"]
        sns.kdeplot(scores, ax=ax, label=category, color=COLORS[category], fill=True, alpha=0.3)
    ax.set_title(version_name)
    ax.set_xlabel("Genuine cosine similarity")
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(DATASET_ROOT, "compare_genuine_distributions.png"), dpi=150)
plt.show()


## 2. EER comparison — Kids vs Adults, per condition

In [ ]:
eer_all = pd.concat([eer[c] for c in CATEGORIES], ignore_index=True)
pivot_eer = eer_all.pivot(index="version", columns="category", values="EER").reindex(VERSIONS)

pivot_eer.plot(kind="bar", figsize=(8, 5), color=[COLORS[c] for c in pivot_eer.columns])
plt.ylabel("Equal Error Rate (EER)")
plt.title("EER by condition: Kids vs Adults")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(DATASET_ROOT, "compare_eer.png"), dpi=150)
plt.show()

pivot_eer


## 3. Score vs. age gap — Kids vs Adults, overlaid (original condition)

In [ ]:
# bins = [5, 8, 11, 14, 17, 20, 30, 40, 50, 60, 70, 80, 90, 100, float("inf")]
# labels = [
#     "5-7", "8-10", "11-13", "14-16", "17-19",
#     "20-29", "30-39", "40-49", "50-59", "60-69",
#     "70-79", "80-89", "90-99", "100+"
# ]

bins = [5,6,7,8,9,10, 11,12,13, 14,15,16, 17,18,19, 20, 30, 40,41,42,43,44,45,46,47,48,49, 50, 60, 70, 80, 90, 100, float("inf")]
labels = [
    "5","6","7","8", "9","10", "11","12","13","14","15", "16", "17","18","19",
    "20-29", "30-39", "40","41","42","43","44","45","46","47","48","", "50-59", "60-69",
    "70-79", "80-89", "90-99", "100+"
    ]




plt.figure(figsize=(8, 5.5))

for category in CATEGORIES:
    df = genuine[category]
    df = df[df["version"] == "original"].copy()

    df["age_gap_bin"] = pd.cut(
        df["age_gap"],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    )

    grouped = df.groupby(
        "age_gap_bin", observed=False
    )["score"].agg(["mean", "std"])

    plt.errorbar(
        grouped.index.astype(str),
        grouped["mean"],
        yerr=grouped["std"],
        marker="o",
        capsize=4,
        label=category,
        color=COLORS[category]
    )

plt.xlabel("Age gap (years)")
plt.ylabel("Mean genuine cosine similarity (original images)")
plt.title("Score vs. age gap: Kids vs Adults")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(DATASET_ROOT, "compare_score_vs_age_gap.png"),
    dpi=150
)

plt.show()


## 4. Is the Kids-vs-Adults difference statistically significant?

A quick **Mann-Whitney U test** (non-parametric, doesn't assume normality) on the genuine
score distributions per condition — useful to state in your report whether the observed gap
between categories is likely real or just sampling noise.


In [ ]:
sig_rows = []
for version_name in VERSIONS:
    kids_scores = genuine["kids"][genuine["kids"]["version"] == version_name]["score"]
    adults_scores = genuine["adults"][genuine["adults"]["version"] == version_name]["score"]
    if len(kids_scores) < 3 or len(adults_scores) < 3:
        continue
    stat, p = mannwhitneyu(kids_scores, adults_scores, alternative="two-sided")
    sig_rows.append({
        "version": version_name,
        "kids_mean": kids_scores.mean(), "adults_mean": adults_scores.mean(),
        "U_stat": stat, "p_value": p, "significant_at_0.05": p < 0.05,
    })

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(os.path.join(DATASET_ROOT, "kids_vs_adults_significance.csv"), index=False)
sig_df


## 5. Summary table: everything in one place

In [ ]:
summary = pivot_eer.copy()
summary.columns = [f"EER_{c}" for c in summary.columns]
summary = summary.merge(
    sig_df.set_index("version")[["kids_mean", "adults_mean", "p_value"]],
    left_index=True, right_index=True, how="left"
)
summary.to_csv(os.path.join(DATASET_ROOT, "final_comparison_summary.csv"))
summary
